# When should an always-on agent wake up?

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/profrodai/profrodai-resources/blob/main/articles/when-should-an-always-on-agent-wake-up/jev_wake_gate_v7.ipynb)

An evaluation notification says: **“All dashboards are green, but a new acceptance case silently drops an evaluation item.”** A small decision model could help an agent decide whether to wake up. Suppose its score for needing attention were **0.08**, and your controller deferred scores at or below **0.20**. Would you let it sleep?

Make a prediction first. Here, the events, labels, scores and USD unit prices are **constructed teaching inputs**. You will run a real Python policy over those inputs, inspect its missed work, change one assumption and write a short engineering recommendation. You will not call Jev or measure any model's accuracy.

This lab accompanies ProfRod's [routing lesson](https://www.profrod.ai/articles/when-should-an-always-on-agent-wake-up), which is held for publication review. The notebook can run independently while that page is unavailable. Basic Python lists and dictionaries are enough; no model-service account is required.


## 1. Open Colab and prepare the experiment

Use the badge above. Connect to a normal **CPU runtime**; a GPU provides no benefit here. Run each section in order, pausing at the prediction questions. **Runtime → Run all** also executes the complete default lab without a prompt, key, upload, package installation or Drive mount. To keep your edits, use **File → Save a copy in Drive**.

The setup cell fetches three small public teaching files from a fixed Git commit, then checks their SHA-256 hashes **before importing the helper**. It never follows a moving `main` branch for executable source. Once fetched, the verified cache works offline. Local Jupyter can also use the original files beside the notebook.

Each setup run creates a new results directory. Rerunning the notebook keeps earlier runs; it does not resume an interrupted computation. Colab runtime storage can disappear, so the final section downloads a results bundle. This is different from durably admitting real incoming work.

By the end, you should be able to explain why zero tuning misses do not certify a policy, audit the denominator of a probability score, price the entire cascade, and distinguish a recorded decision from an accepted event.


In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import json
import os
import platform
import sys
import urllib.request
import uuid

if sys.version_info < (3, 10):
    raise RuntimeError("Use Python 3.10 or newer; no extra packages are needed.")

SOURCE_COMMIT = "9c7a224f9ee334085e480911297276651c9cc18d"
BASE_URL = ("https://raw.githubusercontent.com/profrodai/profrodai-resources/"
            + SOURCE_COMMIT + "/articles/when-should-an-always-on-agent-wake-up/")
ARTIFACT_HASHES = {
    "wake_gate_v3.py": "13556c06a245ac9bab2ac94800113d5d2c929ae534c853c2883d1a878be4c967",
    "synthetic-events-v1.json": "f9f3bad994cde2e946ee43330a84c865ccba17f6718e7a680d8a8d6edb82083a",
    "offline-receipt-v3.json": "00130b2015c7e9e56e1a5a0cb7bffa3d14b3498371d714ed31b964358a9045d8",
}
ASSET_DIR = Path(os.environ.get("PROFROD_LAB_ASSETS", str(Path.cwd() / "profrod-jev-assets")))
ASSET_DIR.mkdir(parents=True, exist_ok=True)
bootstrap = []
for name, expected in ARTIFACT_HASHES.items():
    target = ASSET_DIR / name
    sibling = Path.cwd() / name
    if target.is_file():
        content, origin = target.read_bytes(), "verified cache"
    elif sibling.is_file():
        content, origin = sibling.read_bytes(), "local sibling"
    else:
        try:
            with urllib.request.urlopen(BASE_URL + name, timeout=30) as response:
                content = response.read(1_000_001)
        except OSError as error:
            raise RuntimeError("Could not fetch " + name + ". Check the connection and rerun setup.") from error
        origin = "pinned public download"
    if len(content) > 1_000_000 or hashlib.sha256(content).hexdigest() != expected:
        raise RuntimeError("Changed teaching artifact: " + name + "; no helper was imported. Use a fresh cache directory.")
    if not target.exists():
        with target.open("xb") as destination:
            destination.write(content)
    bootstrap.append({"file": name, "sha256": expected, "origin": origin})

spec = importlib.util.spec_from_file_location("profrod_wake_gate", ASSET_DIR / "wake_gate_v3.py")
lab = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lab)
data = json.loads((ASSET_DIR / "synthetic-events-v1.json").read_text())
events = data["events"]
tune = [event for event in events if event["split"] == "tune"]
test = [event for event in events if event["split"] == "test"]
RUN_ROOT = Path(os.environ.get("PROFROD_LAB_OUTPUT", str(Path.cwd() / "profrod-jev-runs")))
SESSION_DIR = RUN_ROOT / ("notebook-" + uuid.uuid4().hex)
SESSION_DIR.mkdir(parents=True, exist_ok=False)
print("Python:", platform.python_version(), "| source:", SOURCE_COMMIT)
print("Verified files:", [(item["file"], item["origin"]) for item in bootstrap])
print("Results:", SESSION_DIR)


## 2. Write down what you expect

Edit the three strings below before running the next sections. Blank answers do not prevent Run all, but keep your own prediction: it is more useful than agreeing with an explanation afterward.

- Does a low score justify deferring the dropped-item notification?
- If no known-important tuning event is deferred, what does that tell you about **new** events?
- Could paying for cheap judgments increase the total bill?

“Defer” means skip investigation in this replay. It does not implement a timer, an audit or a later wake-up. “Review” means a required route, not a measured human response.


In [ ]:
learner_answers = {
    "initial_notification_decision": "Write your prediction here.",
    "zero_tuning_misses_claim": "Write what this does and does not establish.",
    "cheap_gate_cost_prediction": "Predict which other costs matter.",
    "transfer_recommendation": "Complete the final checkpoint after running the lab.",
}
print("Prediction notes saved for the final export.")


## 3. Select on tuning cases, then freeze

The fixture contains **16 tuning events** and **24 different test episodes**. An episode is one distinct occurrence, not a count of attempts. Their identities do not overlap. The same occurrence retried twice should not become two independent successes.

The controller first respects trusted approval metadata. Otherwise, a malformed score goes to review. A valid score at or below the quiet threshold is deferred; one at or above **0.8** goes to investigation; the middle goes to review. The upper boundary is fixed for this exercise. We select only the lower boundary.

Among five candidates, select the one deferring the most tuning work while missing zero **known-important tuning events**. Ties favor the smaller threshold. This is an exercise rule, not a recommended risk budget. Before running: if 0.30 defers one important tuning event, can it win?


In [ ]:
assert len(tune) == 16 and len(test) == 24
assert {event["episode_id"] for event in tune}.isdisjoint({event["episode_id"] for event in test})
quiet_threshold, tuning_sweep = lab.select_threshold(tune)
print("threshold | tuning deferrals | known-important misses")
for row in tuning_sweep:
    print(f"{row['threshold']:9.2f} | {row['defers']:16d} | {row['important_misses']:22d}")
print("Frozen quiet threshold:", quiet_threshold, "| fixed investigation threshold: 0.8")
assert quiet_threshold == 0.20


## 4. Read the frozen test result

Now run the unchanged original helper. It records the canonical result in a fresh directory and checks malformed responses, protected events, duplicate identity and journal reopening. Later interventions will not replace this result.

Count important episodes deferred, not only the fraction of all events deferred. A route can reduce work and still lose the one event you needed to see. The test labels in this teaching fixture are handwritten; they are not independently observed operational outcomes.


In [ ]:
CANONICAL_DIR = SESSION_DIR / ("canonical-" + uuid.uuid4().hex)
report = lab.run_lab(CANONICAL_DIR)
canonical_before_interventions = json.dumps(report, sort_keys=True)
r = report["results"]["test"]
assert report["threshold_frozen_from_tune"] == quiet_threshold
print("Test episodes:", r["events"], "| routes:", r["routes"])
print("Known-important episodes:", r["important_events"])
print("Important episodes deferred:", r["missed_important_events"])
print("Model calls:", report["model_calls"])
assert r["routes"] == {"defer": 10, "investigate": 2, "review": 12}
assert r["missed_important_events"] == 1 and r["important_events"] == 10


## 5. Find the missed event, then change one decision

The notification from the opening is `test-important-01`. Its constructed score is 0.08, but its expected-attention label is true. The type is valid; the decision is wrong for this known-important case. A parser checking only the probability range cannot detect that error.

**Predict:** lower the quiet threshold from 0.20 to 0.05. Does this case become “investigate,” or does it enter the middle “review” band? What extra work will the other cases create?

Set `INTERVENTION_THRESHOLD` below, then inspect every row of the sweep. This uses test labels to explain a tradeoff. If you choose a revised policy from these rows, you have used these episodes for tuning: they no longer provide untouched evidence for the revised policy. New held-out episodes are required before claiming improvement.


In [ ]:
INTERVENTION_THRESHOLD = 0.05  # Try another value in [0, 0.8), then rerun this cell.
trap = next(event for event in test if event["event_id"] == "test-important-01")
print(trap["event"]["text"])
print("Constructed score:", trap["synthetic_p_yes"], "| known-important label:", trap["expected_attention"])
print("Frozen route:", lab.route(trap, quiet_threshold)[0])
print("Intervention route:", lab.route(trap, INTERVENTION_THRESHOLD)[0])
intervention_result = lab.evaluate(test, INTERVENTION_THRESHOLD)
intervention_sweep = []
print("threshold | defer | investigate | review | important misses | synthetic USD cents")
for threshold in [0.02, 0.05, 0.10, 0.20, 0.30]:
    result = lab.evaluate(test, threshold)
    intervention_sweep.append({"threshold": threshold, "result": result})
    counts = result["routes"]
    print(f"{threshold:9.2f} | {counts['defer']:5d} | {counts['investigate']:11d} | {counts['review']:6d} | {result['missed_important_events']:16d} | {result['synthetic_cascade_cost_cents']:19d}")
assert json.dumps(report, sort_keys=True) == canonical_before_interventions


## 6. Audit the probability-score denominator

For a binary label $y_i$, Brier error is $(p_i-y_i)^2$. Its mean is lower when probabilities better match the binary outcomes, but it is **not a certificate of calibration or a routing safety metric**. A mean can hide an important miss.

Which cases belong in the mean? Here we include only judgments actually attempted, with a valid probability and an unambiguous binary label. Two trusted bypasses never attempt a judgment; two malformed responses provide no valid probability; two ambiguous labels are not binary targets. That leaves **18 of 24** test episodes. They remain in the route/cost table even when excluded from Brier.

The miss denominator is different: **all ten known-important episodes**, including protected and failed-response cases. Do not quietly replace it with the six important episodes entering the Brier calculation. Predict the squared error for the missed event before running: $(0.08-1)^2$.


In [ ]:
decision_by_id = {decision["event_id"]: decision for decision in r["decisions"]}
judgments, excluded = [], []
for event in test:
    decision = decision_by_id[event["event_id"]]
    if not decision["gate_attempted"]:
        reason = "trusted bypass: judgment not attempted"
    elif not lab.valid_probability(event["synthetic_p_yes"]):
        reason = "missing or malformed probability"
    elif type(event["expected_attention"]) is not bool:
        reason = "ambiguous label"
    else:
        error = (event["synthetic_p_yes"] - int(event["expected_attention"])) ** 2
        judgments.append({"event_id": event["event_id"], "squared_error": error})
        continue
    excluded.append({"event_id": event["event_id"], "reason": reason})
brier = sum(item["squared_error"] for item in judgments) / len(judgments)
trap_error = next(item["squared_error"] for item in judgments if item["event_id"] == trap["event_id"])
assert len(judgments) == r["valid_binary_judgments"] == 18
assert abs(brier - r["binary_brier_score"]) < 1e-12
print("Brier:", round(brier, 7), "| judgments:", len(judgments), "| missed-case squared error:", round(trap_error, 4))
print("Excluded from Brier, retained in routes and costs:")
for item in excluded:
    print(item["event_id"], "—", item["reason"])
probability_audit = {"judgments": judgments, "excluded": excluded, "brier": brier}


## 7. Price the workflow, not just the judgment

All prices here are **synthetic USD integer cents**: 1 per gate attempt, 25 per investigation and 50 per review. Trusted bypasses do not pay for a gate; failed attempts still do. The model is just arithmetic; no actual investigator or reviewer outcome is measured.

Predict the total: 22 gate attempts, two investigations and 12 reviews. Compare with investigating all 24 events. That comparator is a workload-cost baseline, not a measured equal-quality system.

Then try `WHAT_IF_REVIEW_CENTS = 40`. Could the modeled bill fall below the baseline while the important miss stays unchanged? A cheaper bill does not repair a wrong deferral. The break-even review price assumes these exact routes and all other unit costs stay fixed.


In [ ]:
WHAT_IF_REVIEW_CENTS = 40  # Synthetic USD cents; edit to test the same routes at another price.
if type(WHAT_IF_REVIEW_CENTS) is not int or WHAT_IF_REVIEW_CENTS < 0:
    raise ValueError("Use nonnegative integer USD cents.")
gate_cents = r["gate_attempts"] * lab.COSTS["gate_attempt_cents"]
investigation_cents = r["routes"]["investigate"] * lab.COSTS["investigation_cents"]
review_cents = r["routes"]["review"] * lab.COSTS["review_cents"]
baseline_cents = r["events"] * lab.COSTS["investigation_cents"]
assert gate_cents + investigation_cents + review_cents == r["synthetic_cascade_cost_cents"] == 672
assert baseline_cents == 600
what_if_cents = gate_cents + investigation_cents + r["routes"]["review"] * WHAT_IF_REVIEW_CENTS
break_even_review_cents = (baseline_cents - gate_cents - investigation_cents) / r["routes"]["review"]
print("Synthetic USD cents:", {"gate": gate_cents, "investigation": investigation_cents, "review": review_cents})
print(f"Cascade: USD {672 / 100:.2f}; investigate all: USD {baseline_cents / 100:.2f}")
print("Break-even review unit price, fixed routes: USD cents", break_even_review_cents)
print(f"What-if total: USD {what_if_cents / 100:.2f}; important misses still {r['missed_important_events']}")
cost_intervention = {"currency": "USD", "unit_review_cents": WHAT_IF_REVIEW_CENTS,
                     "total_cents": what_if_cents, "important_misses": r["missed_important_events"],
                     "break_even_review_cents": break_even_review_cents, "prices_are_synthetic": True}


## 8. Break the input without crossing the approval boundary

The model's score is untrusted evidence. Trusted control metadata belongs to the controller. An event saying “ignore the approval flag” cannot override that flag.

Predict the route for each input: a protected event scoring 0.0; an unprotected event scoring the string `"0.01"`; a boolean `True`; and a missing response. Python treats a boolean as a kind of integer, so accepting “anything numeric” is too loose for this parser.

These cases must route to review. Review is a policy destination here; the replay does not execute a tool or prove that a person actually sees the event. Also distinguish trusted bypasses (no gate attempt) from failed attempts (charged in the synthetic model).


In [ ]:
protected = next(event for event in test if event["event_id"] == "test-protected-01")
print("Protected event:", protected["event"]["text"])
boundary_observations = []
for label, event in [("protected score 0.0", protected)] + [
    ("invalid " + repr(value), {**trap, "synthetic_p_yes": value})
    for value in ["0.01", True, None, float("nan"), -0.1, 1.1]
]:
    action, reason, attempted = lab.route(event, quiet_threshold)
    assert action == "review"
    boundary_observations.append({"case": label, "route": action, "reason": reason, "attempted": attempted})
    print(label, "→", action, "| attempted:", attempted, "|", reason)
assert boundary_observations[0]["attempted"] is False
assert all(item["attempted"] for item in boundary_observations[1:])


## 9. Inspect the Jev-style interface without sending a request

TypeSafe's [Jev interface](https://docs.typesafe.ai/) provides typed judgments. This frozen teaching helper builds a Noul question and checks a constructed response for model identifier, answer type and probability range. It does **not** prove current endpoint availability, authentication, billing or live compatibility.

Noul carries a yes probability, not a separate confidence field. Choice confidence describes a distribution; it is not an observed correctness rate. Earlier classification work includes Yin, Hay and Roth's [NLI approach](https://arxiv.org/abs/1909.00161) and Stepanov and coauthors' [GLiClass](https://arxiv.org/abs/2508.07662). Related ideas and chronology do not establish copying or reveal proprietary model internals. No backend is executed or ranked here.

Inspect the state sent below. It contains event evidence only: no expected label, constructed probability or trusted approval control. If a model sees an answer key, you are no longer measuring the task you intended.


In [ ]:
payload = lab.request_payload(trap)
assert set(payload["state"]) == {"source", "text"}
print(json.dumps(payload, indent=2))
constructed_response = {"model": lab.MODEL_ID, "answers": {
    "needs_attention": {"type": "noul", "noul": 0.08}}}
parsed = lab.parse_jev_response(constructed_response)
assert parsed == 0.08
print("Parsed constructed probability:", parsed, "| network requests to models: 0")
bad_response = {"model": lab.MODEL_ID, "answers": {
    "needs_attention": {"type": "noul", "noul": "0.08"}}}
try:
    lab.parse_jev_response(bad_response)
except ValueError as error:
    print("Malformed typed response rejected:", error)
else:
    raise AssertionError("The parser accepted a string probability.")


## 10. What survives after the decision is written?

Before running, predict how many rows exist at three points: **before deciding**, **after deciding but before recording**, and **after recording but before any downstream action**.

This cell opens a separate SQLite journal and counts rows at those boundaries. It then resubmits the same occurrence, rejects the same identity with conflicting content, closes the connection and reopens it. No process is crashed and no downstream action runs. These are direct observations of the journal, not a claim that real notifications survive an interruption.

The missing property is admission **before** judgment: newly arrived work needs a durable record before it can disappear between steps. Reopening one completed decision does not test that property. Colab's temporary disk is not durable event storage; the downloadable archive is an experiment artifact, not an agent recovery mechanism.


In [ ]:
import sqlite3

JOURNAL_PATH = SESSION_DIR / ("journal-demo-" + uuid.uuid4().hex + ".sqlite3")
connection = sqlite3.connect(JOURNAL_PATH)
try:
    connection.execute("CREATE TABLE inbox(event_id TEXT PRIMARY KEY, payload_hash TEXT NOT NULL, payload TEXT NOT NULL, decision TEXT NOT NULL)")
    count_rows = lambda: connection.execute("SELECT COUNT(*) FROM inbox").fetchone()[0]
    before_deciding = count_rows()
    action, reason, attempted = lab.route(trap, quiet_threshold)
    decision = {"event_id": trap["event_id"], "route": action, "reason": reason, "gate_attempted": attempted}
    after_deciding = count_rows()
    first_write = lab.journal(connection, trap, decision)
    after_recording = count_rows()
    duplicate = lab.journal(connection, trap, decision)
    assert count_rows() == 1
    conflicting = {**trap, "event": {**trap["event"], "text": "different occurrence content"}}
    try:
        lab.journal(connection, conflicting, decision)
    except ValueError:
        conflict_rejected = True
    else:
        raise AssertionError("Conflicting work reused an occurrence identity.")
finally:
    connection.close()
with sqlite3.connect(JOURNAL_PATH) as reopened:
    after_reopening = reopened.execute("SELECT COUNT(*) FROM inbox").fetchone()[0]
journal_observations = {"before_deciding": before_deciding, "after_deciding_before_recording": after_deciding,
    "after_recording_before_action": after_recording, "after_reopening": after_reopening,
    "first_write": first_write, "duplicate": duplicate, "conflict_rejected": conflict_rejected,
    "durable_admission_implemented": False, "downstream_action_executed": False, "process_crash_tested": False}
assert [before_deciding, after_deciding, after_recording, after_reopening] == [0, 0, 1, 1]
print(json.dumps(journal_observations, indent=2))


## 11. Write the recommendation you would hand to another engineer

Return to `learner_answers` in section 2 and replace the final string with a short memo. Rerun that cell with your completed answers, then export below. State:

1. Which important episode was missed, and why a legal probability did not prevent it.
2. Why the Brier denominator is 18 while the important-episode denominator is ten.
3. How a bill below USD 6.00 could coexist with the same miss.
4. Which boundary has no journal row, and what a completed-decision restart check leaves untested.

**Transfer:** suppose a real notification explicitly reports a failed evaluation or a missing expected artifact. Would you let a semantic model decide whether that deterministic condition deserves attention? Keep exact failures, missing artifacts and overdue work in deterministic handling. A later model study could investigate the remaining ambiguous notifications in shadow mode, while existing handling stays authoritative.

To make an operational claim, you would need a real labeled workload, episode-level splits, separately tuned candidate thresholds, severity-separated misses, review time, attention delay, and full costs/latency including failures and retries. No acceptable miss rate can be learned from these 24 constructed examples. Any observed wrongful deferral of critical work blocks enabling that behavior; zero observed misses does not prove zero risk.

Next read [durable work](https://www.profrod.ai/book/ch08-durable-work), [isolation](https://www.profrod.ai/book/ch15-isolation) and [evaluation](https://www.profrod.ai/book/ch16-evaluation). You have a policy experiment and an explanation to inspect, not a new permission to execute actions.


## 12. Save the evidence before the runtime disappears

The bundle contains the untouched canonical receipt/journal, your threshold and price interventions, denominator audit, boundary/journal observations, prediction worksheet, Python/runtime information and the three verified source files. The canonical receipt is kept separate from interventions. Empty answer placeholders are exported honestly; running the cells does not establish that you understood them.

In Colab, this cell starts a browser download with the [official file-download API](https://colab.research.google.com/notebooks/io.ipynb). In local Jupyter it prints the archive path. If the browser blocks the download, use Colab's Files sidebar to download that ZIP. There is no Drive authorization or credential file in this bundle.

If setup reports a network error, reconnect and rerun it. If it reports changed teaching bytes, use a fresh cache path rather than disabling the check. A recycled runtime can reconstruct the pinned inputs, but you must keep the downloaded bundle to retain your previous observations.


In [ ]:
import zipfile

assert json.dumps(report, sort_keys=True) == canonical_before_interventions
exports = {
    "bootstrap-manifest.json": {"source_commit": SOURCE_COMMIT, "artifacts": bootstrap},
    "interventions.json": {"threshold": INTERVENTION_THRESHOLD, "result": intervention_result,
                           "sweep": intervention_sweep, "cost": cost_intervention,
                           "used_test_labels": True, "is_new_held_out_evaluation": False},
    "probability-audit.json": probability_audit,
    "boundary-observations.json": boundary_observations,
    "journal-observations.json": journal_observations,
    "learner-answers.json": learner_answers,
    "runtime.json": {"python": platform.python_version(), "platform": platform.platform(),
                     "source_commit": SOURCE_COMMIT, "model_calls": 0, "actual_model_api_cost_usd": 0.0,
                     "scores_and_unit_prices": "constructed"},
}
EXPORT_DIR = SESSION_DIR / ("export-" + uuid.uuid4().hex)
EXPORT_DIR.mkdir(exist_ok=False)
for name, value in exports.items():
    (EXPORT_DIR / name).write_text(json.dumps(value, indent=2, allow_nan=False) + "\n")
EXPORT_ZIP = SESSION_DIR / ("jev-routing-evidence-" + uuid.uuid4().hex + ".zip")
with zipfile.ZipFile(EXPORT_ZIP, "x", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(EXPORT_DIR.iterdir()):
        archive.write(path, path.name)
    for name in ["receipt.json", "inbox.sqlite3"]:
        archive.write(CANONICAL_DIR / name, "canonical/" + name)
    archive.write(JOURNAL_PATH, "journal-demo.sqlite3")
    for name in ARTIFACT_HASHES:
        archive.write(ASSET_DIR / name, "source/" + name)
print("Saved:", EXPORT_ZIP)
try:
    from google.colab import files
except ImportError:
    print("Local runtime: download or copy this ZIP from the path above.")
else:
    files.download(str(EXPORT_ZIP))
